# CLIP Geometry and Retrieval

Compare queries and returned images. Which visual cue changes the rank?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/openai/clip-vit-base-patch32

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os
import random
import tarfile
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    BertTokenizer, BertModel, BertForMaskedLM,
    CLIPProcessor, CLIPModel, CLIPTokenizerFast,
)
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

In [ ]:
CLIP_ID = "openai/clip-vit-base-patch32"

print(f"📦 Loading CLIP: {CLIP_ID}")
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
clip_model = CLIPModel.from_pretrained(CLIP_ID, revision="refs/pr/15", use_safetensors=True).to(device)
clip_model.eval()

clip_params = sum(p.numel() for p in clip_model.parameters())
print(f"   ✅ Loaded!")
print(f"   - Total parameters: {clip_params / 1e6:.1f}M")
print(f"   - Vision (ViT): {sum(p.numel() for p in clip_model.vision_model.parameters())/1e6:.1f}M")
print(f"   - Text:          {sum(p.numel() for p in clip_model.text_model.parameters())/1e6:.1f}M")
print(f"   - Image size:    {clip_model.config.vision_config.image_size}")
print(f"   - Patch size:    {clip_model.config.vision_config.patch_size}")
print(f"   - Embed dim:     {clip_model.config.projection_dim}")

## Input images

Fixed Oxford-IIIT Pet classroom subset. Source: https://www.robots.ox.ac.uk/~vgg/data/pets/

In [ ]:
from pathlib import Path
import urllib.request, zipfile

IMG_DIR = "./data"
NUM_IMAGES_PER_CLASS = 50
batch_size = 16
if not all(len(list((Path(IMG_DIR) / label).glob("*.jpg"))) == 50 for label in ["dog", "cat"]):
    url = "https://moai.postech.ac.kr/teaching/27s-programming-for-ax/labs/colab/oxford-pets-classroom.zip"
    archive, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(".")
image_paths, image_labels = [], []
for label in ["dog", "cat"]:
    paths = sorted((Path(IMG_DIR) / label).glob("*.jpg"))
    image_paths.extend(str(path) for path in paths)
    image_labels.extend([label] * len(paths))
print(f"Classroom subset: {len(image_paths)} images")


In [ ]:
@torch.no_grad()
def get_clip_image_embeddings(images):
    """Extract image embeddings with the CLIP image encoder."""
    inputs = clip_processor(images=images, return_tensors="pt").to(device)
    embeds = clip_model.get_image_features(**inputs)
    return F.normalize(embeds, p=2, dim=1).cpu().numpy()

@torch.no_grad()
def get_clip_text_embeddings(texts):
    """Extract text embeddings with the CLIP text encoder."""
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True).to(device)
    embeds = clip_model.get_text_features(**inputs)
    return F.normalize(embeds, p=2, dim=1).cpu().numpy()

# Select sample images (4 dogs, 4 cats)
sample_dog_paths = image_paths[:4]
sample_cat_paths = image_paths[NUM_IMAGES_PER_CLASS:NUM_IMAGES_PER_CLASS+4]
sample_paths = sample_dog_paths + sample_cat_paths
sample_images = [Image.open(p).convert('RGB') for p in sample_paths]

# Text prompts
text_prompts = [
    "a photo of a dog",
    "a puppy playing",
    "a photo of a cat",
    "a kitten sleeping",
    "a bird flying",
    "a photo of food",
]

# Extract embeddings
img_embeds = get_clip_image_embeddings(sample_images)
txt_embeds = get_clip_text_embeddings(text_prompts)

# Image-Text Similarity Matrix
similarity = img_embeds @ txt_embeds.T

fig = plt.figure(figsize=(18, 8))
gs = fig.add_gridspec(2, len(sample_images) + 1, width_ratios=[1]*len(sample_images) + [6])

# Image thumbnails (left)
for i, img in enumerate(sample_images):
    ax = fig.add_subplot(gs[i // 4, i % 4])
    ax.imshow(img)
    label = 'Dog' if i < 4 else 'Cat'
    ax.set_title(f'{label} #{i % 4}', fontsize=8)
    ax.axis('off')

# Similarity heatmap (right)
ax_heatmap = fig.add_subplot(gs[:, -1])
im = ax_heatmap.imshow(similarity, cmap='RdYlBu_r', aspect='auto')

img_labels_show = [f"Dog #{i}" for i in range(4)] + [f"Cat #{i}" for i in range(4)]
ax_heatmap.set_yticks(range(len(img_labels_show)))
ax_heatmap.set_yticklabels(img_labels_show, fontsize=10)
ax_heatmap.set_xticks(range(len(text_prompts)))
ax_heatmap.set_xticklabels(text_prompts, rotation=30, ha='right', fontsize=9)
ax_heatmap.set_title('CLIP Image-Text Similarity', fontsize=14, fontweight='bold')

for i in range(len(sample_images)):
    for j in range(len(text_prompts)):
        ax_heatmap.text(j, i, f'{similarity[i,j]:.2f}', ha='center', va='center', fontsize=9,
                       color='white' if similarity[i,j] > 0.25 else 'black')

ax_heatmap.axhline(y=3.5, color='black', linewidth=2)
plt.colorbar(im, ax=ax_heatmap, shrink=0.6)

plt.tight_layout()
plt.show()

print("💡 Observation: dog images have high similarity to 'dog' text; cat images have high similarity to 'cat' text.")
print("   Similarity to unrelated text ('bird', 'food') is low.")

In [ ]:
# Extract image embeddings (full)
print("🔄 Extracting CLIP image embeddings...")
all_img_embeds = []
for i in tqdm(range(0, len(image_paths), batch_size)):
    batch = [Image.open(p).convert('RGB') for p in image_paths[i:i+batch_size]]
    embeds = get_clip_image_embeddings(batch)
    all_img_embeds.append(embeds)
all_img_embeds = np.concatenate(all_img_embeds, axis=0)

# Visualize text embeddings together
mixed_texts = [
    "a photo of a dog", "a puppy", "a golden retriever", "a loyal dog",
    "a photo of a cat", "a kitten", "a tabby cat", "a cute cat",
]
mixed_text_labels = ['dog']*4 + ['cat']*4
text_embeds = get_clip_text_embeddings(mixed_texts)

# Concatenate images + texts and run t-SNE
combined = np.concatenate([all_img_embeds, text_embeds], axis=0)
combined_labels = image_labels + mixed_text_labels
combined_types = ['image'] * len(image_labels) + ['text'] * len(mixed_text_labels)

print("🔄 Computing t-SNE...")
tsne = TSNE(n_components=2, perplexity=min(30, len(combined)-1), random_state=SEED)
tsne_2d = tsne.fit_transform(combined)

print("✅ Done!")

## Demo code

Compare queries and returned images. Which visual cue changes the rank?

In [ ]:
query = "a fluffy white cat"
query_vector = get_clip_text_embeddings([query])
scores = (query_vector @ all_img_embeds.T).ravel()
ranking = np.argsort(scores)[::-1][:5]
print([(image_paths[i], float(scores[i])) for i in ranking])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# (1) Combined image + text t-SNE
for cls, color in [('dog', '#4ECDC4'), ('cat', '#FF6B6B')]:
    # Images (small circle)
    mask = (np.array(combined_labels) == cls) & (np.array(combined_types) == 'image')
    axes[0].scatter(tsne_2d[mask, 0], tsne_2d[mask, 1], c=color, marker='o',
                   s=40, alpha=0.5, edgecolors='white', linewidth=0.3,
                   label=f'{cls} (image)')

    # Texts (large star)
    mask = (np.array(combined_labels) == cls) & (np.array(combined_types) == 'text')
    axes[0].scatter(tsne_2d[mask, 0], tsne_2d[mask, 1], c=color, marker='*',
                   s=400, edgecolors='black', linewidth=1.5, zorder=5,
                   label=f'{cls} (text)')

    # Text labels
    text_mask_indices = np.where(mask)[0]
    text_start = len(image_labels)
    for idx in text_mask_indices:
        text_idx = idx - text_start
        if 0 <= text_idx < len(mixed_texts):
            axes[0].annotate(mixed_texts[text_idx], (tsne_2d[idx, 0], tsne_2d[idx, 1]),
                           fontsize=8, fontweight='bold', textcoords='offset points',
                           xytext=(8, 8), bbox=dict(boxstyle='round,pad=0.2',
                           facecolor='lightyellow', edgecolor=color, alpha=0.9))

axes[0].set_title('CLIP Multimodal Embedding Space\n(Images ○ + Texts ★)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10, loc='best')
axes[0].grid(alpha=0.2)

# (2) Image-only t-SNE
img_only_2d = tsne_2d[:len(image_labels)]
for cls, color, marker in [('dog', '#4ECDC4', 'o'), ('cat', '#FF6B6B', 's')]:
    mask = np.array(image_labels) == cls
    axes[1].scatter(img_only_2d[mask, 0], img_only_2d[mask, 1], c=color, marker=marker,
                   s=60, alpha=0.7, edgecolors='white', linewidth=0.5, label=cls.upper())

axes[1].set_title('CLIP Image Embeddings Only', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=12)
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("💡 Key observations:")
print("   1. Images and texts live in the same space! (the core of CLIP)")
print("   2. Dog text (★) sits near dog images (○), cat text (★) near cat images (○).")
print("   3. This is the principle that makes zero-shot classification work.")

In [ ]:
# Cosine similarity analysis using CLIP image features (compare to Lab 1)
clip_feat_norm = all_img_embeds  # already normalized
clip_sim = cosine_similarity(clip_feat_norm)

labels_arr = np.array(image_labels)
cat_idx = np.where(labels_arr == 'cat')[0]
dog_idx = np.where(labels_arr == 'dog')[0]

cat_cat = clip_sim[np.ix_(cat_idx, cat_idx)]
dog_dog = clip_sim[np.ix_(dog_idx, dog_idx)]
cat_dog = clip_sim[np.ix_(cat_idx, dog_idx)]

cc = cat_cat[np.triu_indices_from(cat_cat, k=1)]
dd = dog_dog[np.triu_indices_from(dog_dog, k=1)]
cd = cat_dog.flatten()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# CLIP similarity distribution
axes[0].hist(cc, bins=25, alpha=0.6, label='Cat↔Cat', color='#FF6B6B', density=True)
axes[0].hist(dd, bins=25, alpha=0.6, label='Dog↔Dog', color='#4ECDC4', density=True)
axes[0].hist(cd, bins=25, alpha=0.6, label='Cat↔Dog', color='#95A5A6', density=True)
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Density')
axes[0].set_title('CLIP Image Feature Similarity', fontsize=13, fontweight='bold')
axes[0].legend()

# CLIP similarity heatmap
sorted_idx = np.concatenate([cat_idx, dog_idx])
sorted_sim = clip_sim[np.ix_(sorted_idx, sorted_idx)]
im = axes[1].imshow(sorted_sim, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[1].axhline(y=len(cat_idx)-0.5, color='black', linewidth=2)
axes[1].axvline(x=len(cat_idx)-0.5, color='black', linewidth=2)
axes[1].set_title('CLIP Cosine Similarity Matrix', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

print(f"\n📊 CLIP Cosine Similarity Statistics:")
print(f"  Cat↔Cat (intra): {cc.mean():.4f} ± {cc.std():.4f}")
print(f"  Dog↔Dog (intra): {dd.mean():.4f} ± {dd.std():.4f}")
print(f"  Cat↔Dog (inter): {cd.mean():.4f} ± {cd.std():.4f}")
print(f"\n  (Compare with the DINO results from Lab 1!)")

In [ ]:
# Demo CLIP's image retrieval — find images using text
print("🔍 CLIP Image Retrieval — searching images with text\n")

search_queries = [
    "a small puppy with big eyes",
    "a fluffy white cat",
    "a large brown dog outdoors",
    "a sleeping cat on furniture",
]

fig, axes = plt.subplots(len(search_queries), 6, figsize=(20, 4 * len(search_queries)))
fig.suptitle('🔍 CLIP Text-to-Image Retrieval', fontsize=16, fontweight='bold')

for row, query in enumerate(search_queries):
    query_embed = get_clip_text_embeddings([query])
    similarities = (query_embed @ all_img_embeds.T).flatten()
    top_indices = similarities.argsort()[::-1][:5]

    # Query text
    axes[row, 0].text(0.5, 0.5, f'"{query}"', ha='center', va='center',
                      fontsize=11, fontweight='bold', wrap=True,
                      transform=axes[row, 0].transAxes,
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='orange'))
    axes[row, 0].set_title('Query', fontsize=11)
    axes[row, 0].axis('off')

    # Top-5 results
    for col, idx in enumerate(top_indices):
        img = Image.open(image_paths[idx]).convert('RGB')
        axes[row, col + 1].imshow(img)
        gt = image_labels[idx].upper()
        axes[row, col + 1].set_title(f'#{col+1} ({gt})\nsim: {similarities[idx]:.3f}', fontsize=9)
        axes[row, col + 1].axis('off')

plt.tight_layout()
plt.show()

## Interpretation

A retrieval rank does not establish the presence of every requested detail.

Compare the output with the evidence in the input. A generated answer can be wrong.